# UVIF Equilibrium with Calibrated Adversarial Gated Fusion for Privacy-Aware Multimodal Emotion Intelligence on IEMOCAP-Light

This notebook implements a real variational intelligence system using the available IEMOCAP-light structure:

- emotion annotations,
- transcriptions,
- sentence-level WAV segments,
- implicit speaker/session identity,
- multimodal utterance structure.

The goal is not to add unnecessary engineering complexity. The goal is to compute four interacting informational forces from real data and study their continuous evolution toward a UVIF equilibrium.

**Focused upgrade included:** adaptive UVIF-gated fusion is added as an active modeling layer.

**Final focused upgrade included:** temperature scaling, calibrated UVIF forces, adversarial privacy projection, coupled UVIF operating objective, and final comparison tables.

## Cell 1 — Environment, reproducibility, and Google Drive folders

In [ ]:

# ============================================================
# Cell 1 — Environment, reproducibility, and Google Drive folders
# ============================================================

from pathlib import Path
import os, re, json, math, random, warnings, subprocess, sys, time
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)

try:
    import numpy as np
    import pandas as pd
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "numpy", "pandas"])
    import numpy as np
    import pandas as pd

np.random.seed(SEED)

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/Outputs/UVIF_IEMOCAP_Equilibrium")
else:
    BASE_DIR = Path.cwd() / "UVIF_IEMOCAP_Equilibrium"

FIG_DIR    = BASE_DIR / "Figures"
TABLE_DIR  = BASE_DIR / "Tables"
MODEL_DIR  = BASE_DIR / "Models"
OUTPUT_DIR = BASE_DIR / "Outputs"
LOG_DIR    = BASE_DIR / "logs"
PYTHON_DIR = BASE_DIR / "Notebook"

for d in [FIG_DIR, TABLE_DIR, MODEL_DIR, OUTPUT_DIR, LOG_DIR, PYTHON_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RUN_LOGS = []
def log(msg):
    stamp = time.strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{stamp}] {msg}"
    print(line)
    RUN_LOGS.append(line)

log(f"Base directory: {BASE_DIR}")
log(f"Colab mode: {IN_COLAB}")


## Cell 2 — Install and import optional scientific libraries

In [ ]:

# ============================================================
# Cell 2 — Install and import optional scientific libraries
# ============================================================

def pip_install(pkg):
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        log(f"Installed: {pkg}")
    except Exception as e:
        log(f"Could not install {pkg}: {e}")

required = ["scikit-learn", "matplotlib", "librosa", "soundfile", "joblib"]
for pkg in required:
    try:
        __import__(pkg.split("==")[0].replace("-", "_"))
    except Exception:
        pip_install(pkg)

# SentenceTransformer is useful but optional.
USE_SENTENCE_TRANSFORMER = True
try:
    from sentence_transformers import SentenceTransformer
except Exception:
    pip_install("sentence-transformers")
    try:
        from sentence_transformers import SentenceTransformer
    except Exception:
        USE_SENTENCE_TRANSFORMER = False

import matplotlib.pyplot as plt
import librosa
import soundfile as sf
from joblib import dump

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, classification_report,
    confusion_matrix, roc_auc_score, log_loss, brier_score_loss
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

log("Scientific libraries loaded.")
log(f"SentenceTransformer available: {USE_SENTENCE_TRANSFORMER}")


## Cell 3 — Dataset paths and automatic file discovery

In [ ]:

# ============================================================
# Cell 3 — Dataset paths and automatic file discovery
# ============================================================

if IN_COLAB:
    DATASET_ROOT = Path("/content/drive/MyDrive/Datasets/IEMOCAP_LIGHT")
else:
    DATASET_ROOT = Path.cwd() / "Datasets" / "IEMOCAP_LIGHT"

EMO_DIR   = DATASET_ROOT / "EmoEvaluation"
TXT_DIR   = DATASET_ROOT / "Transcriptions"
WAV_DIR   = DATASET_ROOT / "Sentences" / "Wav"

for p in [DATASET_ROOT, EMO_DIR, TXT_DIR, WAV_DIR]:
    log(f"{p} exists: {p.exists()}")

emo_files = sorted(list(EMO_DIR.rglob("*.txt"))) if EMO_DIR.exists() else []
txt_files = sorted(list(TXT_DIR.rglob("*.txt"))) if TXT_DIR.exists() else []
wav_files = sorted(list(WAV_DIR.rglob("*.wav"))) if WAV_DIR.exists() else []

log(f"Emotion annotation files found: {len(emo_files)}")
log(f"Transcription files found: {len(txt_files)}")
log(f"Sentence WAV files found: {len(wav_files)}")

pd.DataFrame({
    "component": ["Emotion annotations", "Transcriptions", "Sentence WAV files"],
    "path": [str(EMO_DIR), str(TXT_DIR), str(WAV_DIR)],
    "files_found": [len(emo_files), len(txt_files), len(wav_files)]
}).to_csv(TABLE_DIR / "dataset_file_inventory.csv", index=False)


## Cell 4 — Parse emotion annotations

In [ ]:

# ============================================================
# Cell 4 — Parse emotion annotations
# ============================================================

def parse_emotion_file(path):
    rows = []
    # Typical IEMOCAP line:
    # [6.2901 - 8.2357]	Ses01F_impro01_F000	neu	[2.5000, 2.5000, 2.5000]
    pattern = re.compile(
        r"\[(?P<start>[\d\.]+)\s*-\s*(?P<end>[\d\.]+)\]\s+"
        r"(?P<utt>\S+)\s+(?P<emotion>[a-zA-Z]+)"
    )
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            m = pattern.search(line)
            if m:
                rows.append({
                    "utterance_id": m.group("utt"),
                    "emotion": m.group("emotion"),
                    "start_sec": float(m.group("start")),
                    "end_sec": float(m.group("end")),
                    "source_emotion_file": str(path)
                })
    return rows

emo_rows = []
for f in emo_files:
    emo_rows.extend(parse_emotion_file(f))

emo_df = pd.DataFrame(emo_rows)
if len(emo_df) == 0:
    raise FileNotFoundError("No emotion annotations could be parsed. Please check EmoEvaluation format and path.")

emo_df["duration_sec"] = emo_df["end_sec"] - emo_df["start_sec"]
emo_df["session"] = emo_df["utterance_id"].str.extract(r"(Ses\d+)")
emo_df["speaker"] = emo_df["utterance_id"].str.extract(r"(Ses\d+[FM])")
emo_df["gender_proxy"] = emo_df["speaker"].str[-1]

emo_df.to_csv(TABLE_DIR / "parsed_emotion_annotations.csv", index=False)
log(f"Parsed emotion annotations: {len(emo_df)}")
display(emo_df.head())
display(emo_df["emotion"].value_counts())


## Cell 5 — Parse transcriptions and align by utterance

In [ ]:

# ============================================================
# Cell 5 — Parse transcriptions and align by utterance
# ============================================================

def parse_transcription_file(path):
    rows = []
    # Typical line:
    # Ses01F_impro01_F000 [006.2901-008.2357]: Excuse me.
    pattern = re.compile(r"(?P<utt>Ses\d+[FM]_[^\s]+)\s+\[[^\]]+\]:\s*(?P<text>.*)")
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            m = pattern.search(line.strip())
            if m:
                rows.append({
                    "utterance_id": m.group("utt"),
                    "transcript": m.group("text").strip(),
                    "source_transcription_file": str(path)
                })
    return rows

txt_rows = []
for f in txt_files:
    txt_rows.extend(parse_transcription_file(f))

txt_df = pd.DataFrame(txt_rows)
if len(txt_df) == 0:
    log("Warning: no transcriptions parsed. Text branch will use empty strings.")
    txt_df = pd.DataFrame({"utterance_id": emo_df["utterance_id"], "transcript": ""})

txt_df = txt_df.drop_duplicates("utterance_id")
data_df = emo_df.merge(txt_df, on="utterance_id", how="left")
data_df["transcript"] = data_df["transcript"].fillna("")
data_df["n_words"] = data_df["transcript"].str.split().apply(len)

data_df.to_csv(TABLE_DIR / "aligned_emotion_transcription_table.csv", index=False)
log(f"Aligned emotion + transcription rows: {len(data_df)}")
display(data_df.head())


## Cell 6 — Match utterance IDs to sentence-level WAV files

In [ ]:

# ============================================================
# Cell 6 — Match utterance IDs to sentence-level WAV files
# ============================================================

wav_map = {}
for w in wav_files:
    wav_map[w.stem] = str(w)

data_df["wav_path"] = data_df["utterance_id"].map(wav_map)
matched = data_df["wav_path"].notna().sum()
log(f"Utterances with matched WAV files: {matched} / {len(data_df)}")

# Keep only fully multimodal rows for the main experiment.
mm_df = data_df[data_df["wav_path"].notna()].copy()
if len(mm_df) == 0:
    raise FileNotFoundError("No WAV files matched utterance IDs. Please verify Sentences/Wav organization.")

mm_df.to_csv(TABLE_DIR / "multimodal_aligned_dataset.csv", index=False)
display(mm_df.head())


## Cell 7 — Filter emotion classes for stable modeling

In [ ]:

# ============================================================
# Cell 7 — Filter emotion classes for stable modeling
# ============================================================

# IEMOCAP often contains: neu, hap, exc, sad, ang, fru, fea, sur, dis, oth, xxx.
# For robust lightweight experiments, keep classes with enough samples.
MIN_SAMPLES_PER_CLASS = 20
class_counts = mm_df["emotion"].value_counts()
valid_classes = class_counts[class_counts >= MIN_SAMPLES_PER_CLASS].index.tolist()

# Remove ambiguous/unusable labels if present.
drop_labels = {"xxx", "oth"}
valid_classes = [c for c in valid_classes if c not in drop_labels]

exp_df = mm_df[mm_df["emotion"].isin(valid_classes)].copy()

if exp_df["emotion"].nunique() < 2:
    raise ValueError("Need at least two emotion classes after filtering. Lower MIN_SAMPLES_PER_CLASS.")

log(f"Emotion classes retained: {valid_classes}")
log(f"Experimental utterances: {len(exp_df)}")
display(exp_df["emotion"].value_counts())

exp_df.to_csv(TABLE_DIR / "experimental_dataset_after_filtering.csv", index=False)


## Cell 8 — Text branch: semantic embeddings

In [ ]:

# ============================================================
# Cell 8 — Text branch: semantic embeddings
# ============================================================

texts = exp_df["transcript"].fillna("").astype(str).tolist()

TEXT_EMBEDDING_MODE = "sentence_transformer" if USE_SENTENCE_TRANSFORMER else "tfidf_svd"
TEXT_DIM = 96

if USE_SENTENCE_TRANSFORMER:
    try:
        text_model = SentenceTransformer("all-MiniLM-L6-v2")
        X_text = text_model.encode(texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
        log(f"SentenceTransformer embeddings computed: {X_text.shape}")
    except Exception as e:
        log(f"SentenceTransformer failed, falling back to TF-IDF/SVD: {e}")
        TEXT_EMBEDDING_MODE = "tfidf_svd"
        vectorizer = TfidfVectorizer(max_features=2000, ngram_range=(1,2), min_df=2)
        X_tfidf = vectorizer.fit_transform(texts)
        n_comp = min(TEXT_DIM, max(2, X_tfidf.shape[1]-1))
        svd = TruncatedSVD(n_components=n_comp, random_state=SEED)
        X_text = svd.fit_transform(X_tfidf)
else:
    vectorizer = TfidfVectorizer(max_features=2000, ngram_range=(1,2), min_df=2)
    X_tfidf = vectorizer.fit_transform(texts)
    n_comp = min(TEXT_DIM, max(2, X_tfidf.shape[1]-1))
    svd = TruncatedSVD(n_components=n_comp, random_state=SEED)
    X_text = svd.fit_transform(X_tfidf)
    log(f"TF-IDF/SVD text embeddings computed: {X_text.shape}")

np.save(MODEL_DIR / "X_text.npy", X_text)
log(f"Text embedding mode: {TEXT_EMBEDDING_MODE}")


## Cell 9 — Audio branch: MFCC, spectral, chroma, pitch, and rate features

In [ ]:

# ============================================================
# Cell 9 — Audio branch: MFCC, spectral, chroma, pitch, and rate features
# ============================================================

def safe_audio_features(wav_path, sr_target=16000, max_duration=8.0):
    try:
        y, sr = librosa.load(wav_path, sr=sr_target, mono=True, duration=max_duration)
        if len(y) < sr * 0.05:
            raise ValueError("Very short audio.")
        duration = len(y) / sr

        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
        spec_cent = librosa.feature.spectral_centroid(y=y, sr=sr)
        spec_bw = librosa.feature.spectral_bandwidth(y=y, sr=sr)
        contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
        chroma = librosa.feature.chroma_stft(y=y, sr=sr)
        zcr = librosa.feature.zero_crossing_rate(y)
        rms = librosa.feature.rms(y=y)

        # Pitch/prosody proxy.
        try:
            f0 = librosa.yin(y, fmin=50, fmax=500, sr=sr)
            f0 = f0[np.isfinite(f0)]
            f0_stats = [np.mean(f0), np.std(f0), np.percentile(f0, 25), np.percentile(f0, 75)]
        except Exception:
            f0_stats = [0, 0, 0, 0]

        onset_env = librosa.onset.onset_strength(y=y, sr=sr)
        tempo = librosa.beat.tempo(onset_envelope=onset_env, sr=sr)
        tempo_val = float(tempo[0]) if len(tempo) else 0.0

        def stats(mat):
            return np.concatenate([np.mean(mat, axis=1), np.std(mat, axis=1)])

        feat = np.concatenate([
            stats(mfcc),
            stats(contrast),
            stats(chroma),
            stats(spec_cent),
            stats(spec_bw),
            stats(zcr),
            stats(rms),
            np.array(f0_stats),
            np.array([duration, tempo_val])
        ])
        return feat
    except Exception as e:
        return None

audio_feats, kept_indices, failed = [], [], []
for idx, row in exp_df.iterrows():
    feat = safe_audio_features(row["wav_path"])
    if feat is None or np.any(~np.isfinite(feat)):
        failed.append(row["utterance_id"])
    else:
        audio_feats.append(feat)
        kept_indices.append(idx)

X_audio = np.vstack(audio_feats)
exp_df2 = exp_df.loc[kept_indices].copy().reset_index(drop=True)
X_text2 = X_text[[list(exp_df.index).index(i) for i in kept_indices]]

np.save(MODEL_DIR / "X_audio.npy", X_audio)
exp_df2.to_csv(TABLE_DIR / "final_multimodal_dataset.csv", index=False)

log(f"Audio features computed: {X_audio.shape}")
log(f"Failed audio files: {len(failed)}")
display(exp_df2.head())


## Cell 10 — Labels, identity variables, and train/test split

In [ ]:

# ============================================================
# Cell 10 — Labels, identity variables, and train/test split
# ============================================================

emotion_encoder = LabelEncoder()
speaker_encoder = LabelEncoder()
session_encoder = LabelEncoder()

y_emotion = emotion_encoder.fit_transform(exp_df2["emotion"])
y_speaker = speaker_encoder.fit_transform(exp_df2["speaker"].fillna("unknown"))
y_session = session_encoder.fit_transform(exp_df2["session"].fillna("unknown"))

X_text2 = np.asarray(X_text2)
X_audio = np.asarray(X_audio)
X_multi = np.hstack([X_text2, X_audio])

indices = np.arange(len(exp_df2))

idx_train, idx_test = train_test_split(
    indices,
    test_size=0.25,
    random_state=SEED,
    stratify=y_emotion
)

log(f"Train size: {len(idx_train)} | Test size: {len(idx_test)}")
log(f"Emotion labels: {list(emotion_encoder.classes_)}")
log(f"Speaker labels: {len(speaker_encoder.classes_)} speakers")

pd.DataFrame({
    "emotion_label": emotion_encoder.classes_
}).to_csv(TABLE_DIR / "emotion_label_mapping.csv", index=False)

dump(emotion_encoder, MODEL_DIR / "emotion_label_encoder.joblib")
dump(speaker_encoder, MODEL_DIR / "speaker_label_encoder.joblib")


## Cell 11 — Baseline multimodal emotion classifier

In [ ]:

# ============================================================
# Cell 11 — Baseline multimodal emotion classifier
# ============================================================

emotion_clf = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED))
])

emotion_clf.fit(X_multi[idx_train], y_emotion[idx_train])
proba_test = emotion_clf.predict_proba(X_multi[idx_test])
pred_test = np.argmax(proba_test, axis=1)

emotion_metrics = {
    "accuracy": accuracy_score(y_emotion[idx_test], pred_test),
    "balanced_accuracy": balanced_accuracy_score(y_emotion[idx_test], pred_test),
    "macro_f1": f1_score(y_emotion[idx_test], pred_test, average="macro"),
    "nll": log_loss(y_emotion[idx_test], proba_test, labels=np.arange(len(emotion_encoder.classes_)))
}

log(f"Emotion baseline metrics: {emotion_metrics}")

report = classification_report(
    y_emotion[idx_test], pred_test,
    target_names=emotion_encoder.classes_,
    output_dict=True,
    zero_division=0
)
pd.DataFrame(report).T.to_csv(TABLE_DIR / "emotion_classification_report.csv")
dump(emotion_clf, MODEL_DIR / "baseline_emotion_classifier.joblib")
display(pd.DataFrame([emotion_metrics]))


## Cell 12 — Identity leakage classifier

In [ ]:

# ============================================================
# Cell 12 — Identity leakage classifier
# ============================================================

# Identity leakage is intentionally estimated from the same feature space.
# If speaker identity is easily predictable, privacy pressure should increase.

if len(np.unique(y_speaker)) >= 2:
    speaker_clf = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED))
    ])
    speaker_clf.fit(X_multi[idx_train], y_speaker[idx_train])
    sp_proba = speaker_clf.predict_proba(X_multi[idx_test])
    sp_pred = np.argmax(sp_proba, axis=1)
    identity_metrics = {
        "speaker_accuracy": accuracy_score(y_speaker[idx_test], sp_pred),
        "speaker_balanced_accuracy": balanced_accuracy_score(y_speaker[idx_test], sp_pred),
        "speaker_macro_f1": f1_score(y_speaker[idx_test], sp_pred, average="macro")
    }
    dump(speaker_clf, MODEL_DIR / "identity_leakage_classifier.joblib")
else:
    speaker_clf = None
    identity_metrics = {
        "speaker_accuracy": np.nan,
        "speaker_balanced_accuracy": np.nan,
        "speaker_macro_f1": np.nan
    }

log(f"Identity leakage metrics: {identity_metrics}")
display(pd.DataFrame([identity_metrics]))
pd.DataFrame([identity_metrics]).to_csv(TABLE_DIR / "identity_leakage_metrics.csv", index=False)


## Cell 13 — Calibration, uncertainty, and reliability functions

In [ ]:

# ============================================================
# Cell 13 — Calibration, uncertainty, and reliability functions
# ============================================================

def entropy_from_proba(P, eps=1e-12):
    P = np.clip(P, eps, 1.0)
    return -np.sum(P * np.log(P), axis=1) / np.log(P.shape[1])

def expected_calibration_error(y_true, proba, n_bins=10):
    conf = np.max(proba, axis=1)
    pred = np.argmax(proba, axis=1)
    acc = (pred == y_true).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    bin_rows = []
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        mask = (conf > lo) & (conf <= hi) if i > 0 else (conf >= lo) & (conf <= hi)
        if mask.sum() > 0:
            gap = abs(acc[mask].mean() - conf[mask].mean())
            ece += mask.mean() * gap
            bin_rows.append({
                "bin": i,
                "conf_low": lo,
                "conf_high": hi,
                "count": int(mask.sum()),
                "accuracy": float(acc[mask].mean()),
                "confidence": float(conf[mask].mean()),
                "gap": float(gap)
            })
    return float(ece), pd.DataFrame(bin_rows)

ece, calib_bins = expected_calibration_error(y_emotion[idx_test], proba_test)
uncertainty_test = entropy_from_proba(proba_test)
confidence_test = np.max(proba_test, axis=1)

calibration_metrics = {
    "ece": ece,
    "mean_uncertainty_entropy": float(np.mean(uncertainty_test)),
    "mean_confidence": float(np.mean(confidence_test))
}

log(f"Calibration metrics: {calibration_metrics}")
calib_bins.to_csv(TABLE_DIR / "calibration_bins.csv", index=False)
pd.DataFrame([calibration_metrics]).to_csv(TABLE_DIR / "calibration_metrics.csv", index=False)
display(calib_bins)


## Cell 14 — Operational UVIF forces from real multimodal data

In [ ]:

# ============================================================
# Cell 14 — Operational UVIF forces from real multimodal data
# ============================================================

# Four forces:
# 1. Epistemic pull: high when emotion evidence is certain.
# 2. Privacy pressure: high when identity is predictable.
# 3. Computational drag: high when feature magnitude/dimensional complexity is high.
# 4. Action efficiency: high when calibrated confidence supports robust decisions.

def normalize01(x, eps=1e-12):
    x = np.asarray(x, dtype=float)
    return (x - np.nanmin(x)) / (np.nanmax(x) - np.nanmin(x) + eps)

# Obtain full-data emotion probabilities.
P_emotion_full = emotion_clf.predict_proba(X_multi)
H_emotion = entropy_from_proba(P_emotion_full)
emotion_conf = np.max(P_emotion_full, axis=1)

# Identity leakage probabilities.
if speaker_clf is not None:
    P_speaker_full = speaker_clf.predict_proba(X_multi)
    identity_conf = np.max(P_speaker_full, axis=1)
    H_identity = entropy_from_proba(P_speaker_full)
else:
    identity_conf = np.zeros(len(exp_df2))
    H_identity = np.ones(len(exp_df2))

# Complexity proxy: normalized feature norm and dimensional footprint.
feature_norm = np.linalg.norm(StandardScaler().fit_transform(X_multi), axis=1)
complexity_proxy = normalize01(feature_norm)

# Per-utterance forces.
F_epistemic = normalize01(1.0 - H_emotion)
F_privacy = normalize01(identity_conf)
F_drag = normalize01(complexity_proxy)
F_action = normalize01(emotion_conf * (1.0 - ece))

forces_df = exp_df2[["utterance_id", "emotion", "speaker", "session", "transcript"]].copy()
forces_df["F_epistemic_pull"] = F_epistemic
forces_df["F_privacy_pressure"] = F_privacy
forces_df["F_computational_drag"] = F_drag
forces_df["F_action_efficiency"] = F_action
forces_df["emotion_uncertainty"] = H_emotion
forces_df["emotion_confidence"] = emotion_conf
forces_df["identity_confidence"] = identity_conf

forces_df.to_csv(TABLE_DIR / "uvif_operational_forces_per_utterance.csv", index=False)

force_summary = forces_df[[
    "F_epistemic_pull", "F_privacy_pressure", 
    "F_computational_drag", "F_action_efficiency",
    "emotion_uncertainty", "identity_confidence"
]].describe().T

force_summary.to_csv(TABLE_DIR / "uvif_force_summary.csv")
log("UVIF forces computed from real text/audio/identity evidence.")
display(force_summary)


## Cell 15 — Define the continuous UVIF equilibrium functional

In [ ]:

# ============================================================
# Cell 15 — Define the continuous UVIF equilibrium functional
# ============================================================

# State x = [alpha_text, alpha_audio, privacy_gate, complexity_gate]
# alpha_text and alpha_audio control multimodal evidence allocation.
# privacy_gate suppresses identity leakage.
# complexity_gate controls computational compression.
#
# J_UVIF(x) is minimized when emotional evidence and action efficiency are high,
# while privacy leakage, uncertainty, and complexity are low.

LAMBDA_EPI = 1.00
LAMBDA_ACT = 0.90
LAMBDA_PRIV = 1.10
LAMBDA_DRAG = 0.55
LAMBDA_BAL = 0.20

global_force_means = {
    "epistemic": float(np.mean(F_epistemic)),
    "privacy": float(np.mean(F_privacy)),
    "drag": float(np.mean(F_drag)),
    "action": float(np.mean(F_action)),
    "uncertainty": float(np.mean(H_emotion))
}

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def uvif_terms(x):
    a_text, a_audio, g_priv, g_comp = sigmoid(np.array(x))

    epistemic_gain = (0.55*a_text + 0.45*a_audio) * global_force_means["epistemic"]
    action_gain = (0.50*a_text + 0.50*a_audio) * global_force_means["action"]

    # Privacy gate reduces privacy cost when high.
    privacy_cost = (1.0 - g_priv) * global_force_means["privacy"]

    # Complexity gate reduces drag when high but may over-compress evidence.
    drag_cost = (1.0 - g_comp) * global_force_means["drag"]
    overcompression_cost = 0.15 * g_comp * (1.0 - epistemic_gain)

    # Balance term avoids collapse to one modality.
    balance_cost = (a_text - a_audio)**2

    J = (
        - LAMBDA_EPI * epistemic_gain
        - LAMBDA_ACT * action_gain
        + LAMBDA_PRIV * privacy_cost
        + LAMBDA_DRAG * drag_cost
        + overcompression_cost
        + LAMBDA_BAL * balance_cost
    )

    return {
        "J": float(J),
        "alpha_text": float(a_text),
        "alpha_audio": float(a_audio),
        "privacy_gate": float(g_priv),
        "complexity_gate": float(g_comp),
        "epistemic_gain": float(epistemic_gain),
        "action_gain": float(action_gain),
        "privacy_cost": float(privacy_cost),
        "drag_cost": float(drag_cost),
        "balance_cost": float(balance_cost),
        "overcompression_cost": float(overcompression_cost)
    }

def uvif_objective(x):
    return uvif_terms(x)["J"]

def numerical_gradient(f, x, h=1e-4):
    x = np.asarray(x, dtype=float)
    grad = np.zeros_like(x)
    for i in range(len(x)):
        xp, xm = x.copy(), x.copy()
        xp[i] += h
        xm[i] -= h
        grad[i] = (f(xp) - f(xm)) / (2*h)
    return grad

log(f"Global force means: {global_force_means}")
display(pd.DataFrame([global_force_means]))


## Cell 16 — Continuous equilibrium trajectories

In [ ]:

# ============================================================
# Cell 16 — Continuous equilibrium trajectories
# ============================================================

def run_equilibrium(x0, eta=0.35, steps=120):
    x = np.array(x0, dtype=float)
    rows = []
    for t in range(steps + 1):
        terms = uvif_terms(x)
        terms["t"] = t
        terms["x_raw"] = x.copy().tolist()
        rows.append(terms)
        grad = numerical_gradient(uvif_objective, x)
        x = x - eta * grad
    return pd.DataFrame(rows)

initial_states = {
    "balanced": [0.0, 0.0, 0.0, 0.0],
    "text_dominant": [1.2, -0.8, -0.2, 0.0],
    "audio_dominant": [-0.8, 1.2, -0.2, 0.0],
    "privacy_sensitive": [0.0, 0.0, 1.5, 0.2],
    "low_privacy": [0.0, 0.0, -1.5, 0.0]
}

traj_list = []
for name, x0 in initial_states.items():
    tr = run_equilibrium(x0, eta=0.35, steps=120)
    tr["trajectory"] = name
    traj_list.append(tr)

traj_df = pd.concat(traj_list, ignore_index=True)
traj_df.to_csv(TABLE_DIR / "uvif_continuous_equilibrium_trajectories.csv", index=False)

final_eq = traj_df.sort_values("t").groupby("trajectory").tail(1)
final_eq.to_csv(TABLE_DIR / "uvif_final_equilibrium_states.csv", index=False)

log("Continuous UVIF equilibrium trajectories computed.")
display(final_eq[[
    "trajectory", "J", "alpha_text", "alpha_audio", 
    "privacy_gate", "complexity_gate",
    "epistemic_gain", "action_gain", "privacy_cost", "drag_cost"
]])


## Cell 17 — Figures: force landscape and equilibrium evolution

In [ ]:

# ============================================================
# Cell 17 — Figures: force landscape and equilibrium evolution
# ============================================================

plt.figure(figsize=(8,5))
forces_df[[
    "F_epistemic_pull", "F_privacy_pressure",
    "F_computational_drag", "F_action_efficiency"
]].mean().plot(kind="bar")
plt.ylabel("Mean normalized force")
plt.title("Operational UVIF forces from real multimodal utterances")
plt.tight_layout()
plt.savefig(FIG_DIR / "Fig1_UVIF_Force_Profile.png", dpi=300)
plt.show()

plt.figure(figsize=(8,5))
for name, sub in traj_df.groupby("trajectory"):
    plt.plot(sub["t"], sub["J"], label=name)
plt.xlabel("Equilibrium step")
plt.ylabel("UVIF objective J")
plt.title("Continuous UVIF equilibrium trajectories")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "Fig2_Equilibrium_Objective_Trajectories.png", dpi=300)
plt.show()

plt.figure(figsize=(8,5))
for name, sub in traj_df.groupby("trajectory"):
    plt.plot(sub["t"], sub["privacy_gate"], label=name)
plt.xlabel("Equilibrium step")
plt.ylabel("Privacy gate")
plt.title("Evolution of privacy regulation toward equilibrium")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "Fig3_Privacy_Gate_Evolution.png", dpi=300)
plt.show()

plt.figure(figsize=(8,5))
for name, sub in traj_df.groupby("trajectory"):
    plt.plot(sub["t"], sub["alpha_text"], label=f"{name} text")
    plt.plot(sub["t"], sub["alpha_audio"], linestyle="--", label=f"{name} audio")
plt.xlabel("Equilibrium step")
plt.ylabel("Modality allocation")
plt.title("Text/audio allocation under UVIF equilibrium")
plt.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / "Fig4_Modality_Allocation_Evolution.png", dpi=300)
plt.show()

log("Figures saved.")


## Cell 18 — Utterance-level equilibrium interpretation table

In [ ]:

# ============================================================
# Cell 18 — Utterance-level equilibrium interpretation table
# ============================================================

# Select illustrative utterances where forces conflict strongly.
forces_df["conflict_index"] = (
    forces_df["F_epistemic_pull"]
    + forces_df["F_action_efficiency"]
    + forces_df["F_privacy_pressure"]
    + forces_df["F_computational_drag"]
)

forces_df["semantic_privacy_tension"] = (
    forces_df["F_epistemic_pull"] + forces_df["F_action_efficiency"]
    - forces_df["F_privacy_pressure"] - forces_df["F_computational_drag"]
)

illustrative = forces_df.sort_values("conflict_index", ascending=False).head(20)
illustrative.to_csv(TABLE_DIR / "illustrative_high_conflict_utterances.csv", index=False)

display(illustrative[[
    "utterance_id", "emotion", "speaker", "F_epistemic_pull",
    "F_privacy_pressure", "F_computational_drag", "F_action_efficiency",
    "emotion_uncertainty", "identity_confidence", "transcript"
]].head(10))


## Cell 19 — Ablation: text-only, audio-only, and multimodal decision efficiency

In [ ]:

# ============================================================
# Cell 19 — Ablation: text-only, audio-only, and multimodal decision efficiency
# ============================================================

def eval_model(X, y, idx_train, idx_test, name):
    clf = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED))
    ])
    clf.fit(X[idx_train], y[idx_train])
    P = clf.predict_proba(X[idx_test])
    pred = np.argmax(P, axis=1)
    ece_val, _ = expected_calibration_error(y[idx_test], P)
    out = {
        "model": name,
        "accuracy": accuracy_score(y[idx_test], pred),
        "balanced_accuracy": balanced_accuracy_score(y[idx_test], pred),
        "macro_f1": f1_score(y[idx_test], pred, average="macro"),
        "nll": log_loss(y[idx_test], P, labels=np.arange(len(np.unique(y)))),
        "ece": ece_val,
        "mean_confidence": float(np.max(P, axis=1).mean()),
        "mean_uncertainty": float(entropy_from_proba(P).mean()),
        "feature_dim": X.shape[1]
    }
    return out, clf

ablation_rows = []
text_eval, text_clf = eval_model(X_text2, y_emotion, idx_train, idx_test, "Text only")
audio_eval, audio_clf = eval_model(X_audio, y_emotion, idx_train, idx_test, "Audio only")
multi_eval, multi_clf = eval_model(X_multi, y_emotion, idx_train, idx_test, "Text + Audio")

ablation_rows.extend([text_eval, audio_eval, multi_eval])
ablation_df = pd.DataFrame(ablation_rows)
ablation_df.to_csv(TABLE_DIR / "modality_ablation_results.csv", index=False)

display(ablation_df)

plt.figure(figsize=(8,5))
plt.bar(ablation_df["model"], ablation_df["macro_f1"])
plt.ylabel("Macro-F1")
plt.title("Decision efficiency across modality branches")
plt.tight_layout()
plt.savefig(FIG_DIR / "Fig5_Modality_Ablation_MacroF1.png", dpi=300)
plt.show()

dump(text_clf, MODEL_DIR / "text_only_classifier.joblib")
dump(audio_clf, MODEL_DIR / "audio_only_classifier.joblib")
dump(multi_clf, MODEL_DIR / "multimodal_classifier.joblib")


## Cell 20 — Save outputs summary

In [ ]:

# ============================================================
# Cell 20 — Save outputs summary
# ============================================================

summary_lines = []
summary_lines.append("UVIF Equilibrium for Privacy-Aware Multimodal Emotion Intelligence")
summary_lines.append("=" * 78)
summary_lines.append("")
summary_lines.append("Dataset")
summary_lines.append(f"- Dataset root: {DATASET_ROOT}")
summary_lines.append(f"- Emotion files: {len(emo_files)}")
summary_lines.append(f"- Transcription files: {len(txt_files)}")
summary_lines.append(f"- WAV files: {len(wav_files)}")
summary_lines.append(f"- Final multimodal utterances: {len(exp_df2)}")
summary_lines.append(f"- Emotion classes: {list(emotion_encoder.classes_)}")
summary_lines.append(f"- Speakers: {len(speaker_encoder.classes_)}")
summary_lines.append("")
summary_lines.append("Baseline emotion classifier")
for k, v in emotion_metrics.items():
    summary_lines.append(f"- {k}: {v:.4f}")
summary_lines.append("")
summary_lines.append("Identity leakage")
for k, v in identity_metrics.items():
    try:
        summary_lines.append(f"- {k}: {v:.4f}")
    except Exception:
        summary_lines.append(f"- {k}: {v}")
summary_lines.append("")
summary_lines.append("Calibration")
for k, v in calibration_metrics.items():
    summary_lines.append(f"- {k}: {v:.4f}")
summary_lines.append("")
summary_lines.append("Global UVIF force means")
for k, v in global_force_means.items():
    summary_lines.append(f"- {k}: {v:.4f}")
summary_lines.append("")
summary_lines.append("Final equilibrium states")
summary_lines.append(final_eq[[
    "trajectory", "J", "alpha_text", "alpha_audio",
    "privacy_gate", "complexity_gate", "epistemic_gain",
    "action_gain", "privacy_cost", "drag_cost"
]].to_string(index=False))
summary_lines.append("")
summary_lines.append("Generated files")
for d, label in [(FIG_DIR, "Figures"), (TABLE_DIR, "Tables"), (MODEL_DIR, "Models")]:
    summary_lines.append(f"{label}:")
    for f in sorted(d.glob("*")):
        summary_lines.append(f"- {f.name}")
    summary_lines.append("")
summary_lines.append("Run log")
summary_lines.extend(RUN_LOGS)

summary_text = "\n".join(summary_lines)

# Save in both Outputs and logs for convenience.
(OUTPUT_DIR / "outputs_summary.txt").write_text(summary_text, encoding="utf-8")
(LOG_DIR / "outputs_summary.txt").write_text(summary_text, encoding="utf-8")

log(f"Saved outputs summary to: {OUTPUT_DIR / 'outputs_summary.txt'}")
log(f"Saved outputs summary to: {LOG_DIR / 'outputs_summary.txt'}")

print(summary_text[:4000])


## Cell 21 — Export notebook copy to Google Drive

In [ ]:

# ============================================================
# Cell 21 — Export notebook copy to Google Drive
# ============================================================

# This cell records the intended notebook location.
# In Colab, use File > Save a copy in Drive, or upload this notebook directly.
notebook_manifest = {
    "title": "UVIF_IEMOCAP_Equilibrium_Notebook",
    "base_dir": str(BASE_DIR),
    "fig_dir": str(FIG_DIR),
    "table_dir": str(TABLE_DIR),
    "model_dir": str(MODEL_DIR),
    "output_dir": str(OUTPUT_DIR),
    "log_dir": str(LOG_DIR),
    "python_dir": str(PYTHON_DIR),
    "dataset_root": str(DATASET_ROOT)
}

(Path(PYTHON_DIR) / "notebook_manifest.json").write_text(
    json.dumps(notebook_manifest, indent=2),
    encoding="utf-8"
)

log("Notebook manifest saved.")
display(pd.DataFrame([notebook_manifest]))



## Cell 22 — UVIF-gated fusion: from analysis layer to active intelligence layer

This cell upgrades the notebook from a post-hoc UVIF analysis into an active variational intelligence system.

Instead of using UVIF only after classification, the UVIF forces are used to construct adaptive modality gates:

\[
Z_{\mathrm{UVIF}} =
g_{\mathrm{text}} Z_{\mathrm{text}}
\oplus
g_{\mathrm{audio}} Z_{\mathrm{audio}}
\]

The gates are driven by epistemic pull, privacy pressure, computational drag, uncertainty, and action efficiency.


In [ ]:

# ============================================================
# Cell 22 — UVIF-gated fusion: active force-conditioned representation
# ============================================================

from sklearn.metrics import precision_recall_fscore_support

def make_uvif_gates(F_epi, F_priv, F_drag, F_act, H_unc, temperature=0.75):
    """
    Return adaptive text and audio gates for each utterance.
    Gates are force-conditioned coefficients, not arbitrary neural weights.
    """
    F_epi = np.asarray(F_epi)
    F_priv = np.asarray(F_priv)
    F_drag = np.asarray(F_drag)
    F_act = np.asarray(F_act)
    H_unc = np.asarray(H_unc)

    s_text = (
        0.45 * F_epi +
        0.35 * F_priv +
        0.25 * F_act -
        0.20 * F_drag -
        0.15 * H_unc
    )

    s_audio = (
        0.50 * F_epi +
        0.30 * F_act -
        0.40 * F_priv -
        0.20 * F_drag -
        0.10 * H_unc
    )

    scores = np.vstack([s_text, s_audio]).T / temperature
    scores = scores - scores.max(axis=1, keepdims=True)
    exp_scores = np.exp(scores)
    gates = exp_scores / exp_scores.sum(axis=1, keepdims=True)

    g_text = gates[:, 0]
    g_audio = gates[:, 1]

    shrink = 1.0 - 0.30 * F_drag
    g_text = g_text * shrink
    g_audio = g_audio * shrink

    return g_text, g_audio

g_text, g_audio = make_uvif_gates(
    F_epistemic, F_privacy, F_drag, F_action, H_emotion, temperature=0.75
)

gates_df = exp_df2[["utterance_id", "emotion", "speaker", "session"]].copy()
gates_df["g_text"] = g_text
gates_df["g_audio"] = g_audio
gates_df["gate_sum"] = g_text + g_audio
gates_df["gate_preference"] = np.where(g_text >= g_audio, "text-preferred", "audio-preferred")
gates_df["F_epistemic_pull"] = F_epistemic
gates_df["F_privacy_pressure"] = F_privacy
gates_df["F_computational_drag"] = F_drag
gates_df["F_action_efficiency"] = F_action
gates_df["emotion_uncertainty"] = H_emotion

gates_df.to_csv(TABLE_DIR / "uvif_adaptive_gates_per_utterance.csv", index=False)

log("UVIF adaptive gates computed.")
display(gates_df.head())
display(gates_df[["g_text", "g_audio", "gate_sum"]].describe().T)


## Cell 23 — Build the UVIF-gated multimodal representation

In [ ]:

# ============================================================
# Cell 23 — Build the UVIF-gated multimodal representation
# ============================================================

X_text_scaled = StandardScaler().fit_transform(X_text2)
X_audio_scaled = StandardScaler().fit_transform(X_audio)

X_uvif_gated = np.hstack([
    X_text_scaled * g_text[:, None],
    X_audio_scaled * g_audio[:, None]
])

np.save(MODEL_DIR / "X_uvif_gated.npy", X_uvif_gated)

log(f"UVIF-gated representation shape: {X_uvif_gated.shape}")


## Cell 24 — Compare simple fusion and UVIF-gated fusion

In [ ]:

# ============================================================
# Cell 24 — Compare simple fusion and UVIF-gated fusion
# ============================================================

def evaluate_emotion_model(X, y, idx_train, idx_test, name):
    clf = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2500, class_weight="balanced", random_state=SEED))
    ])
    clf.fit(X[idx_train], y[idx_train])
    P = clf.predict_proba(X[idx_test])
    pred = np.argmax(P, axis=1)
    ece_val, _ = expected_calibration_error(y[idx_test], P)
    H = entropy_from_proba(P)
    return {
        "model": name,
        "accuracy": accuracy_score(y[idx_test], pred),
        "balanced_accuracy": balanced_accuracy_score(y[idx_test], pred),
        "macro_f1": f1_score(y[idx_test], pred, average="macro"),
        "nll": log_loss(y[idx_test], P, labels=np.arange(len(np.unique(y)))),
        "ece": ece_val,
        "mean_confidence": float(np.max(P, axis=1).mean()),
        "mean_uncertainty": float(H.mean()),
        "feature_dim": X.shape[1]
    }, clf, P, pred

comparison_rows = []

eval_text, clf_text_cmp, P_text_cmp, pred_text_cmp = evaluate_emotion_model(
    X_text2, y_emotion, idx_train, idx_test, "Text only"
)
eval_audio, clf_audio_cmp, P_audio_cmp, pred_audio_cmp = evaluate_emotion_model(
    X_audio, y_emotion, idx_train, idx_test, "Audio only"
)
eval_simple, clf_simple_cmp, P_simple_cmp, pred_simple_cmp = evaluate_emotion_model(
    X_multi, y_emotion, idx_train, idx_test, "Simple multimodal fusion"
)
eval_gated, clf_gated_cmp, P_gated_cmp, pred_gated_cmp = evaluate_emotion_model(
    X_uvif_gated, y_emotion, idx_train, idx_test, "UVIF-gated fusion"
)

comparison_rows.extend([eval_text, eval_audio, eval_simple, eval_gated])
uvif_gating_comparison_df = pd.DataFrame(comparison_rows)
uvif_gating_comparison_df.to_csv(TABLE_DIR / "uvif_gated_fusion_comparison.csv", index=False)

dump(clf_gated_cmp, MODEL_DIR / "uvif_gated_emotion_classifier.joblib")

log("UVIF-gated fusion comparison completed.")
display(uvif_gating_comparison_df)


## Cell 25 — Privacy leakage after UVIF gating

In [ ]:

# ============================================================
# Cell 25 — Privacy leakage after UVIF gating
# ============================================================

def evaluate_identity_leakage(X, y_id, idx_train, idx_test, name):
    if len(np.unique(y_id)) < 2:
        return {
            "representation": name,
            "speaker_accuracy": np.nan,
            "speaker_balanced_accuracy": np.nan,
            "speaker_macro_f1": np.nan
        }, None

    clf = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2500, class_weight="balanced", random_state=SEED))
    ])
    clf.fit(X[idx_train], y_id[idx_train])
    P = clf.predict_proba(X[idx_test])
    pred = np.argmax(P, axis=1)
    return {
        "representation": name,
        "speaker_accuracy": accuracy_score(y_id[idx_test], pred),
        "speaker_balanced_accuracy": balanced_accuracy_score(y_id[idx_test], pred),
        "speaker_macro_f1": f1_score(y_id[idx_test], pred, average="macro")
    }, clf

privacy_rows = []
for X_repr, name in [
    (X_text2, "Text only"),
    (X_audio, "Audio only"),
    (X_multi, "Simple multimodal fusion"),
    (X_uvif_gated, "UVIF-gated fusion")
]:
    row, clf_priv = evaluate_identity_leakage(X_repr, y_speaker, idx_train, idx_test, name)
    privacy_rows.append(row)
    if name == "UVIF-gated fusion" and clf_priv is not None:
        dump(clf_priv, MODEL_DIR / "uvif_gated_identity_leakage_classifier.joblib")

privacy_comparison_df = pd.DataFrame(privacy_rows)
privacy_comparison_df.to_csv(TABLE_DIR / "uvif_gated_privacy_leakage_comparison.csv", index=False)

display(privacy_comparison_df)
log("Privacy leakage comparison completed.")


## Cell 26 — Utility–privacy equilibrium score

In [ ]:

# ============================================================
# Cell 26 — Utility–privacy equilibrium score
# ============================================================

score_df = uvif_gating_comparison_df.merge(
    privacy_comparison_df,
    left_on="model",
    right_on="representation",
    how="left"
)

score_df["utility_term"] = normalize01(score_df["macro_f1"].values)
score_df["calibration_term"] = 1.0 - normalize01(score_df["ece"].values)
score_df["uncertainty_term"] = 1.0 - normalize01(score_df["mean_uncertainty"].values)
score_df["privacy_term"] = 1.0 - normalize01(score_df["speaker_accuracy"].fillna(score_df["speaker_accuracy"].mean()).values)

score_df["UVIF_equilibrium_score"] = (
    0.35 * score_df["utility_term"] +
    0.25 * score_df["calibration_term"] +
    0.20 * score_df["uncertainty_term"] +
    0.20 * score_df["privacy_term"]
)

score_df = score_df.sort_values("UVIF_equilibrium_score", ascending=False)
score_df.to_csv(TABLE_DIR / "uvif_utility_privacy_equilibrium_score.csv", index=False)

display(score_df[[
    "model", "macro_f1", "ece", "mean_uncertainty",
    "speaker_accuracy", "UVIF_equilibrium_score"
]])
log("Utility–privacy equilibrium score computed.")


## Cell 27 — Additional figures for the active UVIF-gated model

In [ ]:

# ============================================================
# Cell 27 — Additional figures for the active UVIF-gated model
# ============================================================

plt.figure(figsize=(8,5))
plt.hist(g_text, bins=25, alpha=0.7, label="Text gate")
plt.hist(g_audio, bins=25, alpha=0.7, label="Audio gate")
plt.xlabel("Gate value")
plt.ylabel("Number of utterances")
plt.title("Distribution of UVIF adaptive modality gates")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "Fig6_UVIF_Adaptive_Gate_Distribution.png", dpi=300)
plt.show()

plt.figure(figsize=(8,5))
plt.bar(uvif_gating_comparison_df["model"], uvif_gating_comparison_df["macro_f1"])
plt.ylabel("Macro-F1")
plt.title("Emotion decision performance with UVIF-gated fusion")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(FIG_DIR / "Fig7_UVIF_Gated_Emotion_Performance.png", dpi=300)
plt.show()

plt.figure(figsize=(8,5))
plt.bar(privacy_comparison_df["representation"], privacy_comparison_df["speaker_accuracy"])
plt.ylabel("Speaker identity accuracy")
plt.title("Identity leakage across representations")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(FIG_DIR / "Fig8_UVIF_Gated_Privacy_Leakage.png", dpi=300)
plt.show()

plt.figure(figsize=(8,5))
plt.bar(score_df["model"], score_df["UVIF_equilibrium_score"])
plt.ylabel("Utility–privacy equilibrium score")
plt.title("Overall UVIF equilibrium quality")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(FIG_DIR / "Fig9_UVIF_Utility_Privacy_Equilibrium_Score.png", dpi=300)
plt.show()

log("Additional UVIF-gated figures saved.")


## Cell 28 — Update outputs summary with UVIF-gated fusion results

In [ ]:

# ============================================================
# Cell 28 — Update outputs summary with UVIF-gated fusion results
# ============================================================

extra_summary = []
extra_summary.append("")
extra_summary.append("=" * 78)
extra_summary.append("Focused Upgrade: Active UVIF-Gated Fusion")
extra_summary.append("=" * 78)
extra_summary.append("")
extra_summary.append("This upgrade turns UVIF from a post-hoc force analysis layer into an active")
extra_summary.append("force-conditioned multimodal representation mechanism.")
extra_summary.append("")
extra_summary.append("Adaptive gate statistics")
extra_summary.append(gates_df[["g_text", "g_audio", "gate_sum"]].describe().T.to_string())
extra_summary.append("")
extra_summary.append("Emotion decision comparison")
extra_summary.append(uvif_gating_comparison_df.to_string(index=False))
extra_summary.append("")
extra_summary.append("Privacy leakage comparison")
extra_summary.append(privacy_comparison_df.to_string(index=False))
extra_summary.append("")
extra_summary.append("Utility–privacy equilibrium score")
extra_summary.append(score_df[[
    "model", "macro_f1", "ece", "mean_uncertainty",
    "speaker_accuracy", "UVIF_equilibrium_score"
]].to_string(index=False))
extra_summary.append("")
extra_summary.append("Additional generated files")
extra_summary.append("- Tables/uvif_adaptive_gates_per_utterance.csv")
extra_summary.append("- Tables/uvif_gated_fusion_comparison.csv")
extra_summary.append("- Tables/uvif_gated_privacy_leakage_comparison.csv")
extra_summary.append("- Tables/uvif_utility_privacy_equilibrium_score.csv")
extra_summary.append("- Figures/Fig6_UVIF_Adaptive_Gate_Distribution.png")
extra_summary.append("- Figures/Fig7_UVIF_Gated_Emotion_Performance.png")
extra_summary.append("- Figures/Fig8_UVIF_Gated_Privacy_Leakage.png")
extra_summary.append("- Figures/Fig9_UVIF_Utility_Privacy_Equilibrium_Score.png")
extra_summary.append("- Models/X_uvif_gated.npy")
extra_summary.append("- Models/uvif_gated_emotion_classifier.joblib")
extra_summary.append("- Models/uvif_gated_identity_leakage_classifier.joblib")

extra_text = "\n".join(extra_summary)

summary_path = OUTPUT_DIR / "outputs_summary.txt"
if summary_path.exists():
    previous = summary_path.read_text(encoding="utf-8")
else:
    previous = ""

updated_summary = previous + "\n" + extra_text
summary_path.write_text(updated_summary, encoding="utf-8")
(LOG_DIR / "outputs_summary_gated_upgrade.txt").write_text(updated_summary, encoding="utf-8")

log("Updated outputs summary with active UVIF-gated fusion results.")
print(extra_text)


## Cell 29 — Temperature scaling for calibrated emotion probabilities

The previous notebook reported high calibration error. This cell fits a temperature value on a held-out calibration subset and evaluates calibrated probabilities on a separate evaluation subset.

The calibrated probabilities are then used to recompute epistemic pull, action efficiency, and uncertainty so that UVIF forces are not driven by overconfident raw probabilities.

In [ ]:

# ============================================================
# Cell 29 — Temperature scaling for calibrated emotion probabilities
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

idx_calib, idx_eval = train_test_split(
    idx_test,
    test_size=0.50,
    random_state=SEED,
    stratify=y_emotion[idx_test]
)

base_calib_clf = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2500, class_weight="balanced", random_state=SEED))
])
base_calib_clf.fit(X_multi[idx_train], y_emotion[idx_train])

logits_calib = base_calib_clf.decision_function(X_multi[idx_calib])
logits_eval = base_calib_clf.decision_function(X_multi[idx_eval])
logits_full = base_calib_clf.decision_function(X_multi)

def softmax_np(z):
    z = z - np.max(z, axis=1, keepdims=True)
    e = np.exp(z)
    return e / np.sum(e, axis=1, keepdims=True)

def apply_temperature(logits, T):
    return softmax_np(logits / float(T))

T_grid = np.linspace(0.5, 5.0, 91)
temp_rows = []
for T in T_grid:
    P_cal = apply_temperature(logits_calib, T)
    nll_T = log_loss(y_emotion[idx_calib], P_cal, labels=np.arange(len(emotion_encoder.classes_)))
    ece_T, _ = expected_calibration_error(y_emotion[idx_calib], P_cal)
    temp_rows.append({"T": float(T), "calibration_nll": float(nll_T), "calibration_ece": float(ece_T)})

temp_df = pd.DataFrame(temp_rows)
best_row = temp_df.sort_values("calibration_nll").iloc[0]
T_star = float(best_row["T"])

P_eval_raw = softmax_np(logits_eval)
P_eval_cal = apply_temperature(logits_eval, T_star)

ece_raw_eval, bins_raw_eval = expected_calibration_error(y_emotion[idx_eval], P_eval_raw)
ece_cal_eval, bins_cal_eval = expected_calibration_error(y_emotion[idx_eval], P_eval_cal)

calibration_temperature_results = {
    "T_star": T_star,
    "calibration_nll_at_T_star": float(best_row["calibration_nll"]),
    "calibration_ece_at_T_star": float(best_row["calibration_ece"]),
    "eval_ece_before": float(ece_raw_eval),
    "eval_ece_after": float(ece_cal_eval),
    "eval_nll_before": float(log_loss(y_emotion[idx_eval], P_eval_raw, labels=np.arange(len(emotion_encoder.classes_)))),
    "eval_nll_after": float(log_loss(y_emotion[idx_eval], P_eval_cal, labels=np.arange(len(emotion_encoder.classes_))))
}

temp_df.to_csv(TABLE_DIR / "temperature_scaling_grid.csv", index=False)
pd.DataFrame([calibration_temperature_results]).to_csv(TABLE_DIR / "temperature_scaling_results.csv", index=False)
bins_raw_eval.to_csv(TABLE_DIR / "calibration_bins_before_temperature.csv", index=False)
bins_cal_eval.to_csv(TABLE_DIR / "calibration_bins_after_temperature.csv", index=False)

log(f"Temperature scaling completed. T* = {T_star:.3f}")
log(f"ECE before: {ece_raw_eval:.4f} | ECE after: {ece_cal_eval:.4f}")
display(pd.DataFrame([calibration_temperature_results]))


## Cell 30 — Reliability figure before and after temperature scaling

In [ ]:

# ============================================================
# Cell 30 — Reliability figure before and after temperature scaling
# ============================================================

plt.figure(figsize=(6, 6))
plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
if len(bins_raw_eval) > 0:
    plt.plot(bins_raw_eval["confidence"], bins_raw_eval["accuracy"], marker="o",
             label=f"Before T scaling, ECE={ece_raw_eval:.3f}")
if len(bins_cal_eval) > 0:
    plt.plot(bins_cal_eval["confidence"], bins_cal_eval["accuracy"], marker="o",
             label=f"After T scaling, ECE={ece_cal_eval:.3f}")
plt.xlabel("Confidence")
plt.ylabel("Accuracy")
plt.title("Reliability before and after temperature scaling")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "Fig10_Reliability_Before_After_Temperature.png", dpi=300)
plt.show()

log("Reliability figure saved.")


## Cell 31 — Recompute UVIF forces using calibrated probabilities

This cell recomputes the probability-dependent UVIF forces after calibration. Privacy pressure and drag are preserved as real measured quantities, while epistemic pull and action efficiency are corrected using calibrated probabilities.

In [ ]:

# ============================================================
# Cell 31 — Recompute UVIF forces using calibrated probabilities
# ============================================================

P_emotion_full_raw = softmax_np(logits_full)
P_emotion_full_cal = apply_temperature(logits_full, T_star)

H_emotion_cal = entropy_from_proba(P_emotion_full_cal)
emotion_conf_cal = np.max(P_emotion_full_cal, axis=1)

F_epistemic_cal = normalize01(1.0 - H_emotion_cal)
F_action_cal = normalize01(emotion_conf_cal * (1.0 - ece_cal_eval))
F_privacy_cal = F_privacy.copy()
F_drag_cal = F_drag.copy()

forces_cal_df = forces_df.copy()
forces_cal_df["F_epistemic_pull_uncalibrated"] = forces_df["F_epistemic_pull"]
forces_cal_df["F_action_efficiency_uncalibrated"] = forces_df["F_action_efficiency"]
forces_cal_df["emotion_uncertainty_uncalibrated"] = forces_df["emotion_uncertainty"]
forces_cal_df["emotion_confidence_uncalibrated"] = forces_df["emotion_confidence"]

forces_cal_df["F_epistemic_pull"] = F_epistemic_cal
forces_cal_df["F_action_efficiency"] = F_action_cal
forces_cal_df["emotion_uncertainty"] = H_emotion_cal
forces_cal_df["emotion_confidence"] = emotion_conf_cal

forces_cal_df.to_csv(TABLE_DIR / "uvif_operational_forces_calibrated.csv", index=False)

calibrated_force_summary = forces_cal_df[[
    "F_epistemic_pull",
    "F_privacy_pressure",
    "F_computational_drag",
    "F_action_efficiency",
    "emotion_uncertainty",
    "identity_confidence"
]].describe().T

calibrated_force_summary.to_csv(TABLE_DIR / "uvif_force_summary_calibrated.csv")

global_force_means_cal = {
    "epistemic": float(np.mean(F_epistemic_cal)),
    "privacy": float(np.mean(F_privacy_cal)),
    "drag": float(np.mean(F_drag_cal)),
    "action": float(np.mean(F_action_cal)),
    "uncertainty": float(np.mean(H_emotion_cal))
}

log(f"Calibrated global force means: {global_force_means_cal}")
display(calibrated_force_summary)


## Cell 32 — Calibrated UVIF gate construction

In [ ]:

# ============================================================
# Cell 32 — Calibrated UVIF gate construction
# ============================================================

g_text_cal, g_audio_cal = make_uvif_gates(
    F_epistemic_cal,
    F_privacy_cal,
    F_drag_cal,
    F_action_cal,
    H_emotion_cal,
    temperature=0.75
)

gates_cal_df = exp_df2[["utterance_id", "emotion", "speaker", "session"]].copy()
gates_cal_df["g_text_calibrated"] = g_text_cal
gates_cal_df["g_audio_calibrated"] = g_audio_cal
gates_cal_df["gate_sum_calibrated"] = g_text_cal + g_audio_cal
gates_cal_df["gate_preference_calibrated"] = np.where(g_text_cal >= g_audio_cal, "text-preferred", "audio-preferred")

gates_cal_df.to_csv(TABLE_DIR / "uvif_calibrated_adaptive_gates_per_utterance.csv", index=False)

X_uvif_cal_gated = np.hstack([
    X_text_scaled * g_text_cal[:, None],
    X_audio_scaled * g_audio_cal[:, None]
])
np.save(MODEL_DIR / "X_uvif_calibrated_gated.npy", X_uvif_cal_gated)

log("Calibrated UVIF gates and representation computed.")
display(gates_cal_df[["g_text_calibrated", "g_audio_calibrated", "gate_sum_calibrated"]].describe().T)


## Cell 33 — Adversarial privacy by speaker-direction projection

This cell implements a lightweight adversarial privacy approximation without requiring PyTorch.  
A speaker classifier is trained on the calibrated UVIF-gated representation, then its discriminative direction is removed with increasing strength. This directly tests whether speaker prediction can be made harder while preserving emotion utility.

In [ ]:

# ============================================================
# Cell 33 — Adversarial privacy by speaker-direction projection
# ============================================================

def remove_speaker_direction(X, y_speaker, idx_train, lambdas):
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)

    sp = LogisticRegression(max_iter=2500, class_weight="balanced", random_state=SEED)
    sp.fit(Xs[idx_train], y_speaker[idx_train])

    W = sp.coef_
    if W.ndim == 2 and W.shape[0] == 1:
        direction = W[0]
    else:
        direction = np.mean(W, axis=0)

    u = direction / (np.linalg.norm(direction) + 1e-12)

    Xs_priv_list = {}
    for lam in lambdas:
        projection = Xs @ u
        Xs_priv = Xs - float(lam) * projection[:, None] * u[None, :]
        Xs_priv_list[float(lam)] = Xs_priv

    return Xs_priv_list, sp, u

privacy_lambdas = [0.0, 0.25, 0.50, 0.75, 1.00, 1.25, 1.50]
X_priv_dict, speaker_head_for_projection, speaker_direction = remove_speaker_direction(
    X_uvif_cal_gated,
    y_speaker,
    idx_train,
    privacy_lambdas
)

np.save(MODEL_DIR / "uvif_speaker_direction.npy", speaker_direction)

log("Speaker-predictive direction estimated and privacy-projected representations generated.")


## Cell 34 — Privacy–utility Pareto sweep

In [ ]:

# ============================================================
# Cell 34 — Privacy–utility Pareto sweep
# ============================================================

def eval_privacy_utility(X, lam_name):
    emotion_eval, emotion_model, P_eval_model, pred_eval_model = evaluate_emotion_model(
        X, y_emotion, idx_train, idx_test, f"Adversarial UVIF lambda={lam_name}"
    )
    privacy_eval, privacy_model = evaluate_identity_leakage(
        X, y_speaker, idx_train, idx_test, f"Adversarial UVIF lambda={lam_name}"
    )
    row = {}
    row.update(emotion_eval)
    row.update({
        "privacy_lambda": float(lam_name),
        "speaker_accuracy": privacy_eval["speaker_accuracy"],
        "speaker_balanced_accuracy": privacy_eval["speaker_balanced_accuracy"],
        "speaker_macro_f1": privacy_eval["speaker_macro_f1"]
    })
    return row, emotion_model, privacy_model

pareto_rows = []
best_models = {}
for lam, Xp in X_priv_dict.items():
    row, emo_m, priv_m = eval_privacy_utility(Xp, lam)
    pareto_rows.append(row)
    best_models[float(lam)] = (emo_m, priv_m)

uvif_privacy_pareto_df = pd.DataFrame(pareto_rows)
uvif_privacy_pareto_df["utility_norm"] = normalize01(uvif_privacy_pareto_df["macro_f1"].values)
uvif_privacy_pareto_df["privacy_norm"] = 1.0 - normalize01(uvif_privacy_pareto_df["speaker_accuracy"].values)
uvif_privacy_pareto_df["calibration_norm"] = 1.0 - normalize01(uvif_privacy_pareto_df["ece"].values)
uvif_privacy_pareto_df["adversarial_equilibrium_score"] = (
    0.45 * uvif_privacy_pareto_df["utility_norm"] +
    0.35 * uvif_privacy_pareto_df["privacy_norm"] +
    0.20 * uvif_privacy_pareto_df["calibration_norm"]
)

uvif_privacy_pareto_df.to_csv(TABLE_DIR / "uvif_adversarial_privacy_pareto_scored.csv", index=False)

best_adv_row = uvif_privacy_pareto_df.sort_values("adversarial_equilibrium_score", ascending=False).iloc[0]
best_lambda = float(best_adv_row["privacy_lambda"])
best_adv_emotion_model, best_adv_privacy_model = best_models[best_lambda]

dump(best_adv_emotion_model, MODEL_DIR / "best_adversarial_uvif_emotion_classifier.joblib")
dump(best_adv_privacy_model, MODEL_DIR / "best_adversarial_uvif_identity_classifier.joblib")
np.save(MODEL_DIR / "X_best_adversarial_uvif.npy", X_priv_dict[best_lambda])

log(f"Best adversarial UVIF lambda: {best_lambda}")
display(uvif_privacy_pareto_df)


## Cell 35 — Coupled UVIF operating objective

This cell explicitly couples utility, privacy, calibration, uncertainty, and complexity into one operating objective. Raw metrics are still reported separately; the coupled objective is only used to select a scientifically interpretable operating point from the Pareto frontier.

In [ ]:

# ============================================================
# Cell 35 — Coupled UVIF operating objective
# ============================================================

df_coupled = uvif_privacy_pareto_df.copy()
df_coupled["J_emotion"] = 1.0 - df_coupled["macro_f1"]
df_coupled["J_privacy"] = df_coupled["speaker_accuracy"]
df_coupled["J_calibration"] = df_coupled["ece"]
df_coupled["J_uncertainty"] = df_coupled["mean_uncertainty"]
df_coupled["J_complexity"] = df_coupled["feature_dim"] / df_coupled["feature_dim"].max()

L_UTIL = 1.00
L_PRIV = 0.75
L_CAL = 0.50
L_UNC = 0.20
L_COMP = 0.05

df_coupled["J_UVIF_coupled"] = (
    L_UTIL * df_coupled["J_emotion"] +
    L_PRIV * df_coupled["J_privacy"] +
    L_CAL * df_coupled["J_calibration"] +
    L_UNC * df_coupled["J_uncertainty"] +
    L_COMP * df_coupled["J_complexity"]
)

df_coupled = df_coupled.sort_values("J_UVIF_coupled", ascending=True)
best_coupled = df_coupled.iloc[0]
best_coupled_lambda = float(best_coupled["privacy_lambda"])

df_coupled.to_csv(TABLE_DIR / "uvif_coupled_objective_operating_points.csv", index=False)

log("Coupled UVIF operating objective computed.")
display(df_coupled[[
    "privacy_lambda", "macro_f1", "speaker_accuracy", "ece",
    "mean_uncertainty", "J_UVIF_coupled", "adversarial_equilibrium_score"
]])


## Cell 36 — Figures for calibration, adversarial privacy, and coupled objective

In [ ]:

# ============================================================
# Cell 36 — Figures for calibration, adversarial privacy, and coupled objective
# ============================================================

plt.figure(figsize=(8,5))
plt.plot(temp_df["T"], temp_df["calibration_nll"], marker="o", markersize=3)
plt.axvline(T_star, linestyle="--", label=f"T*={T_star:.2f}")
plt.xlabel("Temperature T")
plt.ylabel("Calibration NLL")
plt.title("Temperature scaling objective")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "Fig11_Temperature_Scaling_NLL.png", dpi=300)
plt.show()

plt.figure(figsize=(8,5))
plt.plot(uvif_privacy_pareto_df["speaker_accuracy"], uvif_privacy_pareto_df["macro_f1"], marker="o")
for _, r in uvif_privacy_pareto_df.iterrows():
    plt.annotate(f"λ={r['privacy_lambda']}", (r["speaker_accuracy"], r["macro_f1"]), fontsize=8)
plt.xlabel("Speaker leakage accuracy")
plt.ylabel("Emotion macro-F1")
plt.title("Privacy–utility Pareto frontier")
plt.tight_layout()
plt.savefig(FIG_DIR / "Fig12_Privacy_Utility_Pareto.png", dpi=300)
plt.show()

plt.figure(figsize=(8,5))
plt.plot(df_coupled["privacy_lambda"], df_coupled["J_UVIF_coupled"], marker="o")
plt.xlabel("Privacy adversarial strength lambda")
plt.ylabel("Coupled UVIF objective")
plt.title("Coupled UVIF operating objective")
plt.tight_layout()
plt.savefig(FIG_DIR / "Fig13_Coupled_UVIF_Objective.png", dpi=300)
plt.show()

plt.figure(figsize=(8,5))
plt.bar(["Before temperature", "After temperature"], [ece_raw_eval, ece_cal_eval])
plt.ylabel("ECE")
plt.title("Calibration error reduction")
plt.tight_layout()
plt.savefig(FIG_DIR / "Fig14_ECE_Before_After_Temperature.png", dpi=300)
plt.show()

log("Upgrade figures saved.")


## Cell 37 — Unified final comparison table

In [ ]:

# ============================================================
# Cell 37 — Unified final comparison table
# ============================================================

final_rows = []

def safe_add_model_row(label, X):
    e, _, _, _ = evaluate_emotion_model(X, y_emotion, idx_train, idx_test, label)
    p, _ = evaluate_identity_leakage(X, y_speaker, idx_train, idx_test, label)
    final_rows.append({
        "model": label,
        "macro_f1": e["macro_f1"],
        "balanced_accuracy": e["balanced_accuracy"],
        "accuracy": e["accuracy"],
        "nll": e["nll"],
        "ece": e["ece"],
        "mean_uncertainty": e["mean_uncertainty"],
        "speaker_accuracy": p["speaker_accuracy"],
        "feature_dim": e["feature_dim"]
    })

safe_add_model_row("Text only", X_text2)
safe_add_model_row("Audio only", X_audio)
safe_add_model_row("Simple multimodal fusion", X_multi)

if "X_uvif_gated" in globals():
    safe_add_model_row("Original UVIF-gated fusion", X_uvif_gated)

safe_add_model_row("Calibrated UVIF-gated fusion", X_uvif_cal_gated)
safe_add_model_row(f"Adversarial calibrated UVIF lambda={best_lambda}", X_priv_dict[best_lambda])
safe_add_model_row(f"Coupled-selected UVIF lambda={best_coupled_lambda}", X_priv_dict[best_coupled_lambda])

final_comparison_df = pd.DataFrame(final_rows)
simple_f1 = final_comparison_df.loc[final_comparison_df["model"] == "Simple multimodal fusion", "macro_f1"].iloc[0]
simple_leakage = final_comparison_df.loc[final_comparison_df["model"] == "Simple multimodal fusion", "speaker_accuracy"].iloc[0]

final_comparison_df["privacy_gain_vs_simple"] = simple_leakage - final_comparison_df["speaker_accuracy"]
final_comparison_df["macro_f1_delta_vs_simple"] = final_comparison_df["macro_f1"] - simple_f1

final_comparison_df.to_csv(TABLE_DIR / "final_uvif_upgrade_comparison.csv", index=False)

display(final_comparison_df)
log("Unified final comparison table saved.")


## Cell 38 — Updated outputs summary for final upgrade

In [ ]:

# ============================================================
# Cell 38 — Updated outputs summary for final upgrade
# ============================================================

final_upgrade_summary = []
final_upgrade_summary.append("")
final_upgrade_summary.append("=" * 78)
final_upgrade_summary.append("Final Upgrade: Calibrated + Adversarial + Coupled UVIF Gating")
final_upgrade_summary.append("=" * 78)
final_upgrade_summary.append("")
final_upgrade_summary.append("1. Temperature scaling")
final_upgrade_summary.append(pd.DataFrame([calibration_temperature_results]).to_string(index=False))
final_upgrade_summary.append("")
final_upgrade_summary.append("2. Calibrated force means")
for k, v in global_force_means_cal.items():
    final_upgrade_summary.append(f"- {k}: {v:.4f}")
final_upgrade_summary.append("")
final_upgrade_summary.append("3. Calibrated gate statistics")
final_upgrade_summary.append(
    gates_cal_df[["g_text_calibrated", "g_audio_calibrated", "gate_sum_calibrated"]].describe().T.to_string()
)
final_upgrade_summary.append("")
final_upgrade_summary.append("4. Adversarial privacy Pareto frontier")
final_upgrade_summary.append(uvif_privacy_pareto_df.to_string(index=False))
final_upgrade_summary.append("")
final_upgrade_summary.append("5. Coupled UVIF operating objective")
final_upgrade_summary.append(df_coupled[[
    "privacy_lambda", "macro_f1", "speaker_accuracy", "ece",
    "mean_uncertainty", "J_UVIF_coupled", "adversarial_equilibrium_score"
]].to_string(index=False))
final_upgrade_summary.append("")
final_upgrade_summary.append("6. Final comparison")
final_upgrade_summary.append(final_comparison_df.to_string(index=False))
final_upgrade_summary.append("")
final_upgrade_summary.append("Generated upgrade files")
for f in [
    "Tables/temperature_scaling_grid.csv",
    "Tables/temperature_scaling_results.csv",
    "Tables/calibration_bins_before_temperature.csv",
    "Tables/calibration_bins_after_temperature.csv",
    "Tables/uvif_operational_forces_calibrated.csv",
    "Tables/uvif_force_summary_calibrated.csv",
    "Tables/uvif_calibrated_adaptive_gates_per_utterance.csv",
    "Tables/uvif_adversarial_privacy_pareto_scored.csv",
    "Tables/uvif_coupled_objective_operating_points.csv",
    "Tables/final_uvif_upgrade_comparison.csv",
    "Figures/Fig10_Reliability_Before_After_Temperature.png",
    "Figures/Fig11_Temperature_Scaling_NLL.png",
    "Figures/Fig12_Privacy_Utility_Pareto.png",
    "Figures/Fig13_Coupled_UVIF_Objective.png",
    "Figures/Fig14_ECE_Before_After_Temperature.png",
    "Models/X_uvif_calibrated_gated.npy",
    "Models/uvif_speaker_direction.npy",
    "Models/X_best_adversarial_uvif.npy",
    "Models/best_adversarial_uvif_emotion_classifier.joblib",
    "Models/best_adversarial_uvif_identity_classifier.joblib"
]:
    final_upgrade_summary.append(f"- {f}")

final_upgrade_text = "\n".join(final_upgrade_summary)

summary_path = OUTPUT_DIR / "outputs_summary.txt"
previous_summary = summary_path.read_text(encoding="utf-8") if summary_path.exists() else ""
summary_path.write_text(previous_summary + "\n" + final_upgrade_text, encoding="utf-8")
(LOG_DIR / "outputs_summary_final_upgrade.txt").write_text(
    previous_summary + "\n" + final_upgrade_text,
    encoding="utf-8"
)

log("Final upgrade summary saved.")
print(final_upgrade_text)
